# code de modéle

Ce code importe plusieurs modules de la bibliothèque PyTorch, qui est un framework populaire pour la construction et l'entraînement de réseaux neuronaux.

Voici un bref résumé de ce que chaque module importé fait :

- `torch`: Fournit la fonctionnalité principale pour travailler avec des tenseurs (tableaux multidimensionnels) et construire des graphes de calcul.
- `torch.nn`: Contient des classes pour construire différents types de couches et de modèles de réseau neuronal.
- `torchvision.datasets`: Fournit un accès à des ensembles de données d'images populaires, tels que MNIST et CIFAR-10.
- `torchvision.transforms`: Contient diverses transformations d'images, telles que le recadrage et le redimensionnement.
- `torch.autograd.Variable`: Fournit un moyen de calculer les gradients par rapport aux tenseurs qui ont requires_grad défini sur True.

Dans l'ensemble, ce code fait probablement partie d'un programme plus important qui utilise PyTorch pour construire et entraîner un réseau neuronal pour la classification d'images.

In [2]:
import torch
import torch.nn as nn
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torch.autograd import Variable

Ces variables définissent les hyperparamètres du modèle de réseau neuronal qui sera construit et entraîné. Voici une brève explication de chaque variable:

- `input_size`: La taille de la couche d'entrée du réseau, qui dépend de la taille des images en entrée. Dans ce cas, les images sont de taille 28x28, donc la couche d'entrée a une taille de 28.
- `hidden_size`: La taille de chaque couche cachée du réseau. Dans ce cas, chaque couche cachée a une taille de 128.
- `num_layers`: Le nombre de couches cachées dans le réseau. Dans ce cas, il y a 2 couches cachées.
- `num_classes`: Le nombre de classes de sortie du réseau. Dans ce cas, il y a 10 classes de sortie correspondant aux chiffres de 0 à 9.
- `batch_size`: Le nombre d'images qui sont traitées simultanément dans chaque itération de l'entraînement.
- `num_epochs`: Le nombre d'itérations complètes à travers l'ensemble de données d'entraînement pendant l'entraînement.
- `learning_rate`: Le taux d'apprentissage utilisé pour mettre à jour les poids du réseau pendant l'entraînement.

In [2]:
# Hyper Parameters 
input_size = 28
hidden_size = 128
num_layers = 2
num_classes = 10
batch_size = 100
num_epochs = 5
learning_rate = 0.001

Ce code définit les ensembles de données d'entraînement et de test pour le modèle de réseau neuronal. Dans ce cas, les ensembles de données MNIST sont utilisés, qui sont des images de chiffres de 0 à 9 de taille 28x28.

- `train_dataset`: L'ensemble de données d'entraînement MNIST, qui est téléchargé depuis Internet et stocké dans un dossier local appelé "data". Les images sont transformées en tenseurs à l'aide de la fonction `transforms.ToTensor()`.
- `test_dataset`: L'ensemble de données de test MNIST, qui est également téléchargé depuis Internet et stocké dans le même dossier local. Les images sont également transformées en tenseurs à l'aide de la fonction `transforms.ToTensor()`.

Ensuite, le code crée des chargeurs de données pour l'ensemble de données d'entraînement et de test. Les chargeurs de données sont utilisés pour charger les données en lots (batch) dans le modèle de réseau neuronal pendant l'entraînement et les tests.

- `train_loader`: Le chargeur de données pour l'ensemble de données d'entraînement. Il charge les données en lots de taille `batch_size` et les mélange à chaque itération d'entraînement.
- `test_loader`: Le chargeur de données pour l'ensemble de données de test. Il charge également les données en lots de taille `batch_size`, mais ne les mélange pas car l'ordre n'a pas d'importance lors des tests.

In [3]:
# MNIST dataset 
train_dataset = dsets.MNIST(root='./data', 
                            train=True, 
                            transform=transforms.ToTensor(),  
                            download=True)

test_dataset = dsets.MNIST(root='./data', 
                           train=False, 
                           transform=transforms.ToTensor())

# Data Loader (Input Pipeline)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, 
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset, 
                                          batch_size=batch_size, 
                                          shuffle=False)

Ce code définit la classe `GRNN`, qui est une sous-classe de `nn.Module` de PyTorch. Cette classe représente le modèle de réseau neuronal GRU (Gated Recurrent Unit) qui sera utilisé pour la classification d'images.

La méthode `__init__()` est appelée lors de la création d'une instance de la classe `GRNN`. Elle définit les couches du réseau et les paramètres nécessaires pour les initialiser. Dans ce cas, le modèle GRU est composé d'une couche d'entrée, de `num_layers` couches cachées GRU, et d'une couche de sortie linéaire (`fc`) qui produit des scores de classe pour chaque image.

La méthode `forward()` est appelée lorsque les données sont propagées à travers le réseau. Elle définit la séquence d'opérations à effectuer pour calculer les sorties du réseau à partir des entrées `x`. Dans ce cas, la méthode effectue une propagation en avant de la couche GRU sur les données d'entrée `x`, suivi d'une couche linéaire pour produire des scores de classe pour chaque image.

Enfin, une instance de la classe `GRNN` est créée avec les hyperparamètres précédemment définis et stockée dans la variable `grnn_model`. Cette instance représente le modèle de réseau neuronal qui sera entraîné et utilisé pour la classification d'images.

In [4]:
# GRNN Model
class GRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(GRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True) #type de couche réccurente
        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        # Set initial states
        h0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size))

        # Forward propagate GRU
        out, _ = self.gru(x, h0)

        # Decode hidden state of last time step
        out = self.fc(out[:, -1, :])
        return out

grnn_model = GRNN(input_size, hidden_size, num_layers, num_classes)

Ce code charge un modèle de réseau neuronal pré-entraîné à partir d'un fichier enregistré sous forme de fichier `pth`. 

La fonction `torch.load()` est utilisée pour charger les poids du modèle à partir du fichier spécifié. Le paramètre `map_location` est utilisé pour spécifier que les poids doivent être chargés sur le CPU, même si le modèle a été entraîné sur un GPU.

Ensuite, les poids sont appliqués à l'instance de la classe `GRNN` stockée dans la variable `grnn_model`. La méthode `load_state_dict()` est utilisée pour charger les poids dans le modèle. Cependant, les noms des clés des poids dans le fichier enregistré peuvent ne pas correspondre exactement aux noms des paramètres dans le modèle. Pour résoudre ce problème, le code définit un dictionnaire `mapping` pour faire correspondre les noms des clés dans le fichier enregistré aux noms des paramètres dans le modèle. Ensuite, le code utilise une compréhension de dictionnaire pour charger les poids en utilisant les noms de clé correspondants dans le dictionnaire `mapping`.

In [5]:
# Load the saved Model
state_dict = torch.load('grnn_model.pth', map_location=torch.device('cpu'))
mapping = {'gru.weight_ih_l0': 'gru.weight_ih_l0', 'gru.weight_hh_l0': 'gru.weight_hh_l0', 'gru.bias_ih_l0': 'gru.bias_ih_l0', 'gru.bias_hh_l0': 'gru.bias_hh_l0', 'gru.weight_ih_l1': 'gru.weight_ih_l1', 'gru.weight_hh_l1': 'gru.weight_hh_l1', 'gru.bias_ih_l1': 'gru.bias_ih_l1', 'gru.bias_hh_l1': 'gru.bias_hh_l1', 'fc.weight': 'fc.weight', 'fc.bias': 'fc.bias'}
grnn_model.load_state_dict({mapping[k]: v for k, v in state_dict.items() if k in mapping})


<All keys matched successfully>

Ce code définit la fonction de perte et l'optimiseur pour le modèle de réseau neuronal.

- `nn.CrossEntropyLoss()`: La fonction de perte à utiliser pour l'entraînement du modèle. Dans ce cas, la fonction de perte de l'entropie croisée est utilisée pour mesurer la différence entre les scores de classe prédits par le modèle et les vraies étiquettes de classe des images.
- `torch.optim.Adam()`: L'optimiseur utilisé pour mettre à jour les poids du modèle pendant l'entraînement. Dans ce cas, l'optimiseur Adam est utilisé avec un taux d'apprentissage (`lr`) de `learning_rate`. L'optimiseur Adam est une méthode d'optimisation qui adapte le taux d'apprentissage pour chaque paramètre en fonction des estimations des moments du gradient.

In [6]:
# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(grnn_model.parameters(), lr=learning_rate)

Ce code entraîne le modèle de réseau neuronal `grnn_model` en utilisant l'ensemble de données d'entraînement.

Le code utilise deux boucles `for` imbriquées pour parcourir les images et les étiquettes de l'ensemble de données d'entraînement par lots (`batch`) à chaque itération.

À chaque itération, les images sont d'abord transformées en tenseurs PyTorch et stockées dans la variable `images`. Les étiquettes sont également stockées dans la variable `labels`. 

Ensuite, les poids du modèle sont initialisés à zéro à l'aide de la méthode `zero_grad()` de l'optimiseur. Les images sont ensuite propagées à travers le modèle en appelant la méthode `forward()` du modèle sur `images` pour obtenir les scores de classe prédits, stockés dans la variable `outputs`. La fonction de perte est ensuite calculée en appelant `criterion(outputs, labels)`.

La méthode `backward()` est ensuite appelée sur la fonction de perte pour calculer les gradients de tous les paramètres du modèle par rapport à la perte. Enfin, l'optimiseur est utilisé pour mettre à jour les poids du modèle en appelant la méthode `step()`.

Le code affiche également la perte à chaque 100ème itération pour suivre la progression de l'entraînement. La boucle extérieure `for` parcourt les époques d'entraînement, et la boucle intérieure `for` parcourt les itérations dans chaque époque.

In [7]:
# Train the Model
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = Variable(images.view(-1, 28, 28))
        labels = Variable(labels)

        # Forward + Backward + Optimize
        optimizer.zero_grad()
        outputs = grnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if (i+1) % 100 == 0:
            print ('Epoch [%d/%d], Step [%d/%d], Loss: %.4f' 
                   %(epoch+1, num_epochs, i+1, len(train_dataset)//batch_size, loss.data))

Epoch [1/5], Step [100/600], Loss: 0.0006
Epoch [1/5], Step [200/600], Loss: 0.0036
Epoch [1/5], Step [300/600], Loss: 0.0094
Epoch [1/5], Step [400/600], Loss: 0.0009
Epoch [1/5], Step [500/600], Loss: 0.0441
Epoch [1/5], Step [600/600], Loss: 0.0190
Epoch [2/5], Step [100/600], Loss: 0.0031
Epoch [2/5], Step [200/600], Loss: 0.0445
Epoch [2/5], Step [300/600], Loss: 0.0038
Epoch [2/5], Step [400/600], Loss: 0.0150
Epoch [2/5], Step [500/600], Loss: 0.0043
Epoch [2/5], Step [600/600], Loss: 0.0026
Epoch [3/5], Step [100/600], Loss: 0.0002
Epoch [3/5], Step [200/600], Loss: 0.0008
Epoch [3/5], Step [300/600], Loss: 0.0098
Epoch [3/5], Step [400/600], Loss: 0.0019
Epoch [3/5], Step [500/600], Loss: 0.0002
Epoch [3/5], Step [600/600], Loss: 0.0042
Epoch [4/5], Step [100/600], Loss: 0.0004
Epoch [4/5], Step [200/600], Loss: 0.0030
Epoch [4/5], Step [300/600], Loss: 0.0360
Epoch [4/5], Step [400/600], Loss: 0.0067
Epoch [4/5], Step [500/600], Loss: 0.0072
Epoch [4/5], Step [600/600], Loss:

Ce code teste la précision du modèle de réseau neuronal `grnn_model` sur l'ensemble de données de test.

Le code utilise une boucle `for` pour parcourir les images et les étiquettes de l'ensemble de données de test. À chaque itération, les images sont transformées en tenseurs PyTorch et stockées dans la variable `images`. Les scores de classe prédits sont ensuite calculés en appelant la méthode `forward()` sur `images` pour obtenir les scores de classe prédits, stockés dans la variable `outputs`.

La méthode `torch.max()` est ensuite utilisée pour obtenir les indices des classes prédites avec les scores de classe prédits les plus élevés. Ces indices sont stockés dans la variable `predicted`.

La boucle suit également le nombre total d'images testées (`total`) et le nombre d'images correctement classées (`correct`).

Enfin, le code calcule et affiche la précision finale du modèle en divisant le nombre d'images correctement classées par le nombre total d'images dans l'ensemble de données de test et en multipliant le résultat par 100 pour obtenir un pourcentage.

In [8]:
# Test the Model
correct = 0
total = 0
for images, labels in test_loader:
    images = Variable(images.view(-1, 28, 28))
    outputs = grnn_model(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum()

print('Accuracy of the network on the 10000 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 10000 test images: 99 %


Ce code enregistre les poids du modèle de réseau neuronal `grnn_model` dans un fichier `pth` pour une utilisation ultérieure.

La fonction `torch.save()` est utilisée pour enregistrer les poids du modèle dans le fichier spécifié. Les poids sont extraits à partir de l'instance de la classe `GRNN` en appelant la méthode `state_dict()` sur l'instance.

Les poids sont enregistrés dans le fichier spécifié sous forme de dictionnaire Python, avec les noms des paramètres du modèle comme clés et les valeurs des paramètres comme valeurs associées. Le fichier enregistré peut être utilisé pour recharger les poids du modèle dans une autre instance de la classe `GRNN` pour effectuer des prédictions sur de nouvelles données.

In [9]:
# Save the Model
torch.save(grnn_model.state_dict(), 'grnn_model.pth')

# Test

Ce code importe plusieurs bibliothèques Python pour effectuer des tâches spécifiques:

- `cv2`: La bibliothèque OpenCV, qui est utilisée pour le traitement d'images et la vision par ordinateur.
- `numpy`: La bibliothèque NumPy, qui est utilisée pour les calculs numériques et la manipulation de tableaux de données.
- `pytesseract`: La bibliothèque PyTesseract, qui est utilisée pour la reconnaissance optique de caractères (OCR).
- `tkinter`: La bibliothèque Tkinter, qui est utilisée pour créer des interfaces graphiques utilisateur (GUI).
- `filedialog` et `ImageTk` de Tkinter, ainsi que `PIL`: Ces bibliothèques sont utilisées pour afficher des images et ouvrir des boîtes de dialogue pour sélectionner des fichiers.
- `torch` et `nn`: Ces bibliothèques sont utilisées pour construire et entraîner des modèles de réseau neuronal à l'aide de PyTorch.
- `googletrans`: La bibliothèque Googletrans, qui est utilisée pour la traduction de texte.
Ces deux lignes de code Python permettent de travailler avec des fonctionnalités de synthèse vocale.

-la bibliothèque `gTTS`, qui permet de convertir du texte en discours.

-la bibliothèque `pyttsx3`, qui fournit une interface pour le moteur de synthèse vocale eSpeak. Cela permet de produire des sons directement à partir du code Python sans avoir besoin d'un fichier audio préenregistré.

-La bibliothèque `os` fournit des fonctions pour interagir avec le système d'exploitation sous-jacent. Dans le contexte de ce code, la fonction "os" est utilisée pour exécuter des commandes système, comme la lecture d'un fichier audio.

-La bibliothèque `langdetect` fournit une fonction appelée detect() qui permet de détecter automatiquement la langue d'un texte donné.

In [10]:
import cv2
import numpy as np
import pytesseract
import tkinter as tk
from tkinter import filedialog
from PIL import ImageTk, Image
import torch
import torch.nn as nn
from googletrans import Translator
from gtts import gTTS
import os
import pyttsx3
from langdetect import detect

Ce code définit la classe `GRNN` pour un modèle de réseau neuronal GRU (Gated Recurrent Unit) pour la classification de chiffres manuscrits. Le modèle prend en entrée des images de taille 28x28 pixels.

La méthode `__init__()` est appelée lors de la création d'une instance de la classe `GRNN`. Elle définit les couches du réseau et les paramètres nécessaires pour les initialiser. Dans ce cas, le modèle GRU est composé d'une couche d'entrée de taille 28, de deux couches cachées GRU de taille 128, et d'une couche de sortie linéaire (`fc`) qui produit des scores de classe pour chaque image.

La méthode `forward()` est appelée lorsque les données sont propagées à travers le réseau. Elle définit la séquence d'opérations à effectuer pour calculer les sorties du réseau à partir des entrées `x`. Dans ce cas, la méthode effectue une propagation en avant de la couche GRU sur les données d'entrée `x`, suivi d'une couche linéaire pour produire des scores de classe pour chaque image.

L'initialisation des paramètres de la couche cachée GRU est effectuée par la méthode `torch.zeros()` qui crée un tenseur de zéros de la taille requise. La méthode `to()` est utilisée pour spécifier que les tenseurs doivent être stockés sur le CPU.

Cette classe définit un modèle GRU de base pour la classification de chiffres manuscrits. Cependant, il est probable que le modèle soit entraîné sur des données supplémentaires pour améliorer ses performances.

In [11]:
# Define the GRNN model
class GRNN(nn.Module):
    def __init__(self):
        super(GRNN, self).__init__()
        self.gru = nn.GRU(input_size=28, hidden_size=128, num_layers=2, batch_first=True)
        self.fc = nn.Linear(128, 10)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 128, device='cpu')
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

Ce code charge les poids pré-entraînés d'un modèle de réseau neuronal GRU (Gated Recurrent Unit) pour la classification de chiffres manuscrits à partir d'un fichier enregistré.

Une instance de la classe `GRNN` est créée en appelant son constructeur sans argument, puis les poids du modèle sont chargés à partir du fichier spécifié en utilisant la méthode `load_state_dict()`. Le paramètre `map_location` est utilisé pour spécifier que les poids doivent être chargés sur le CPU, même si le modèle a été entraîné sur un GPU.

Enfin, la méthode `eval()` est appelée sur le modèle pour le mettre en mode d'évaluation. Cette méthode désactive certaines couches du modèle qui sont utilisées uniquement pendant l'entraînement, comme la couche de normalisation par lots (`BatchNorm`), pour que le modèle puisse être utilisé pour la prédiction.

In [12]:
# Load the GRNN model
grnn_model = GRNN()
grnn_model.load_state_dict(torch.load('grnn_model.pth', map_location=torch.device('cpu')))
grnn_model.eval()

GRNN(
  (gru): GRU(28, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)

La fonction "init()" initialise une instance du moteur de synthèse vocale. Elle peut prendre des arguments facultatifs pour personnaliser le moteur, tels que la voix à utiliser, le taux de parole, le volume, etc.

Une fois que l'instance du moteur est créée, vous pouvez l'utiliser pour convertir le texte en discours en appelant la fonction "say()" et en passant le texte que vous souhaitez convertir en argument. Ensuite, vous pouvez appeler la fonction "runAndWait()" pour attendre que la conversion soit effectuée et que le discours soit prononcé.

In [ ]:
# Initialize the text-to-speech engine
engine = pyttsx3.init()

Cette fonction prend en entrée le chemin d'une image, traite l'image pour extraire le texte à l'aide de Tesseract OCR, et prédit la classe de chiffre manuscrit contenu dans l'image en utilisant le modèle de réseau neuronal GRU entraîné.

Le code utilise la bibliothèque PyTesseract pour extraire le texte de l'image. La méthode `image_to_string()` est utilisée pour extraire le texte à partir de l'image spécifiée. Le langage de l'image est spécifié comme anglais et français.

Ensuite, le code utilise la bibliothèque OpenCV pour charger l'image, la convertir en niveaux de gris et la redimensionner pour la passer en entrée au modèle GRU. L'image est également normalisée avec les mêmes valeurs utilisées lors de l'entraînement du modèle.

L'image est ensuite convertie en un tenseur PyTorch et passée dans le modèle GRU à l'aide de la méthode `forward()`. La classe de chiffre manuscrit prédite est extraite en utilisant la méthode `argmax()` pour trouver l'indice de la sortie ayant la valeur la plus élevée, puis convertie en une valeur entière à l'aide de la méthode `item()`.

Enfin, la fonction retourne le texte extrait de l'image et la classe de chiffre manuscrit prédite sous forme de chaîne de caractères.

In [13]:
# Define a function to process the image and extract the text using Tesseract OCR and GRNN
def process_image(image_path):
    # Extract the text from the image using Tesseract OCR
    pytesseract.pytesseract.tesseract_cmd = r'C:\\Program Files\\Tesseract-OCR\\tesseract.exe'
    text = pytesseract.image_to_string(Image.open(image_path), lang='eng+fr')

    # Convert the image to grayscale and resize it for GRNN input
    img = cv2.imread(image_path, 0)
    img = cv2.resize(img, (28, 28))

    # Normalize the image with the same values used during training
    img = (img.astype(np.float32) / 255.0 - 0.1307) / 0.3081

    # Reshape the image for GRNN input
    img = np.reshape(img, (1, 28, 28))

    # Convert the image to a tensor and pass it through the GRNN model
    img = torch.from_numpy(img)
    output = grnn_model(img)

    # Convert the output to text
    output = output.squeeze().argmax().item()

    return text + " " + str(output)

Cette fonction crée une boîte de dialogue pour permettre à l'utilisateur de sélectionner une image à partir de son système de fichiers.

Le code utilise la bibliothèque Tkinter pour créer une boîte de dialogue `askopenfilename()` qui permet à l'utilisateur de sélectionner un fichier. La variable `file_path` est utilisée pour stocker le chemin du fichier sélectionné.

Ensuite, le code utilise la bibliothèque PIL pour ouvrir l'image sélectionnée à l'aide de `Image.open()`. L'image est redimensionnée pour tenir dans la zone de visualisation de l'interface graphique et affichée à l'aide de la méthode `configure()` de `image_label`.

In [14]:
# Define a function to open a file dialog and import an image
def open_file_dialog(event=None):
    # Create a file dialog and allow the user to select a file
    global file_path
    file_path = filedialog.askopenfilename()

    # Display the selected image in the GUI
    img = Image.open(file_path)
    img = img.resize((400, 400))
    img = ImageTk.PhotoImage(img)
    image_label.configure(image=img)
    image_label.image = img

Bien sûr ! Cette fonction `show_text()` affiche le texte extrait de l'image dans la zone de texte de sortie, en détectant également la langue du texte grâce à la bibliothèque `langdetect`.

Voici comment cela fonctionne :

1. Tout d'abord, la fonction `process_image()` est appelée pour extraire le texte de l'image sélectionnée.

2. Ensuite, la fonction `detect()` de la bibliothèque `langdetect` est appelée pour détecter automatiquement la langue du texte extrait.

3. Ensuite, la variable `target_lang` est initialisée à `None` pour indiquer que le texte n'a pas encore été traduit.

4. Ensuite, la fonction vérifie le contenu de la zone de texte de sortie pour déterminer la langue cible éventuelle (anglais, français ou arabe) préalablement sélectionnée par l'utilisateur.

5. Ensuite, la zone de texte de sortie est effacée et le texte extrait est inséré dans la zone de texte de sortie. Enfin, la langue détectée est ajoutée à la fin du texte extrait dans la zone de texte de sortie.

Cela permet à l'utilisateur de voir le texte extrait et sa langue détectée, et de décider s'il souhaite traduire le texte dans une autre langue ou le lire à haute voix.

In [15]:
# Define a function to show the extracted text
def show_text():
    # Call process_image to extract the text from the image
    text = process_image(file_path)
    # Detect the language of the extracted text
    lang = detect(text)
    # Set the output text variable to display the extracted text
    global target_lang
    target_lang = None
    if 'ar' in output_text.get('1.0', tk.END):
        target_lang = 'ar'
    elif 'fr' in output_text.get('1.0', tk.END):
        target_lang = 'fr'
    elif 'en' in output_text.get('1.0', tk.END):
        target_lang = 'en'
    output_text.delete('1.0', tk.END)
    output_text.insert(tk.END, text)
    output_text.insert(tk.END, f"{text} ({lang})")


Cette fonction gère l'événement "WM_DELETE_WINDOW" qui se produit lorsque l'utilisateur clique sur le bouton de fermeture de la fenêtre de l'interface graphique.

Si l'utilisateur clique sur le bouton "OK" dans la boîte de dialogue de confirmation, la méthode `destroy()` est appelée sur l'objet `root` pour fermer l'interface graphique.

Le code utilise également la méthode `askokcancel()` de Tkinter pour afficher une boîte de dialogue de confirmation demandant à l'utilisateur s'il veut vraiment quitter. Si l'utilisateur clique sur le bouton "Annuler", la méthode `askokcancel()` renvoie `False` et la fenêtre reste ouverte.

In [16]:
# Define a function to handle the "WM_DELETE_WINDOW" event
def on_closing():
    if tk.messagebox.askokcancel("Quitter", "Voulez-vous vraiment quitter ?"):
        root.destroy()

Cette fonction Python définit une fonction appelée "translate_text(dest)" qui traduit le texte extrait en une langue spécifiée.

Pour traduire le texte, la fonction "process_image()" est d'abord appelée pour extraire le texte de l'image en utilisant le chemin du fichier image stocké dans la variable "file_path". Le texte extrait est stocké dans la variable "text".

Ensuite, la bibliothèque "googletrans" est utilisée pour traduire le texte extrait dans la langue spécifiée. La variable "dest" contient le code de langue de la langue de destination. La fonction "translate()" de l'objet "translator" est appelée en passant le texte extrait et le code de langue de destination en arguments. Le texte traduit est stocké dans la variable "translated_text".

Enfin, la fonction met à jour la variable "target_lang" avec le code de langue de destination, efface le contenu actuel de la zone de texte de sortie (objet "output_text") et insère le texte traduit à la fin de cette zone.

En résumé, cette fonction extrait le texte de l'image, le traduit dans la langue spécifiée en utilisant la bibliothèque "googletrans" et affiche le texte traduit dans la zone de texte de sortie.

In [17]:
# Define a function to translate the extracted text to the specified language
def translate_text(dest):
    # Call process_image to extract the text from the image
    text = process_image(file_path)

    # Translate the text to the specified language
    translator = Translator()
    translated_text = translator.translate(text, dest=dest).text

    # Set the output text variable to display the translated text
    global target_lang
    target_lang = dest
    output_text.delete('1.0', tk.END)
    output_text.insert(tk.END, translated_text)

Cette fonction `read_text()` lit le texte extrait ou traduit à haute voix en utilisant la bibliothèque `gTTS`. Voici comment cela fonctionne :

1. Tout d'abord, la fonction vérifie si une langue cible a été sélectionnée en vérifiant la valeur de la variable `target_lang`. Si `target_lang` est `None`, cela signifie qu'aucune langue cible n'a été sélectionnée, et un message d'erreur est affiché à l'utilisateur.

2. Si une langue cible a été sélectionnée, la fonction `process_image()` est appelée pour extraire le texte de l'image sélectionnée.

3. Ensuite, si la langue cible est différente de l'anglais, la fonction utilise la bibliothèque `googletrans` pour traduire le texte dans la langue cible.

4. Enfin, la fonction utilise la bibliothèque `gTTS` pour générer un fichier audio contenant la lecture à haute voix du texte extrait ou traduit, puis utilise la commande système `os.system()` pour lancer la lecture de ce fichier audio.

Cela permet à l'utilisateur d'écouter le texte extrait ou traduit à haute voix pour une meilleure compréhension ou une utilisation pratique.

In [ ]:
# Define a function to read the text aloud
def read_text():
    global target_lang
    if target_lang is None:
        messagebox.showerror("Erreur", "Veuillez d'abord afficher ou traduire le texte")
        return

    # Call process_image to extract the text from the image
    text = process_image(file_path)

    # Translate the text to the target language, if necessary
    if target_lang != 'en':
        translator = Translator()
        text = translator.translate(text, dest=target_lang).text

    # Read the text aloud using gTTS
    tts = gTTS(text=text, lang=target_lang)
    tts.save("temp.mp3")
    os.system("start temp.mp3")

Ce code crée une interface utilisateur graphique (GUI) pour l'application OCR. Voici ce que chaque élément de l'interface utilisateur fait :

1. La première ligne crée une fenêtre principale avec le titre "OCR avec Tesseract OCR et GRNN".

2. La deuxième ligne lie l'événement "WM_DELETE_WINDOW" à la fonction `on_closing()`, qui gère la fermeture de la fenêtre.

3. La troisième ligne crée une étiquette pour afficher l'image sélectionnée.

4. La quatrième ligne crée un bouton pour ouvrir la boîte de dialogue de sélection de fichier et importer une image.

5. La cinquième ligne crée un bouton pour afficher le texte extrait de l'image sélectionnée.

6. Les lignes suivantes créent des boutons pour traduire le texte extrait dans différentes langues : arabe, français et anglais.

7. La ligne suivante crée un bouton pour lire le texte extrait ou traduit à haute voix.

8. Les lignes suivantes créent une barre de défilement et une zone de texte pour afficher le texte extrait ou traduit.

9. La ligne suivante lie la touche "q" à la fonction `on_closing()`, qui permet de fermer l'application en appuyant sur la touche "q".

10. Enfin, la dernière ligne démarre la boucle principale de l'interface utilisateur graphique (GUI).

Cela permet à l'utilisateur d'interagir avec l'application OCR en sélectionnant une image, en extrayant le texte de l'image, en le traduisant dans différentes langues et en le lisant à haute voix. Le texte extrait ou traduit est affiché dans la zone de texte, où l'utilisateur peut le copier ou le coller selon ses besoins.

In [1]:
# Create the GUI
root = tk.Tk()
root.title("OCR avec Tesseract OCR et GRNN")

# Bind the "WM_DELETE_WINDOW" event to the on_closing function
root.protocol("WM_DELETE_WINDOW", on_closing)

# Create a label to display the selected image
image_label = tk.Label(root)
image_label.pack()

# Create a button to open the file dialog and import an image
import_button = tk.Button(root, text="Importer une image", command=open_file_dialog)
import_button.pack()

# Create a button to show the extracted text
show_text_button = tk.Button(root, text="Afficher le texte", command=show_text)
show_text_button.pack()

# Create buttons to translate the extracted text to different languages
translate_to_arabic_button = tk.Button(root, text="Traduire en arabe", command=lambda: translate_text('ar'))
translate_to_arabic_button.pack()

translate_to_french_button = tk.Button(root, text="Traduire en français", command=lambda: translate_text('fr'))
translate_to_french_button.pack()

translate_to_english_button = tk.Button(root, text="Translate to English", command=lambda: translate_text('en'))
translate_to_english_button.pack()

# Create a button to read the text aloud
read_text_button = tk.Button(root, text="Lire le texte", command=read_text)
read_text_button.pack()

# Create a scrollbar and a text box to display the output text
scrollbar = tk.Scrollbar(root)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

output_text = tk.Text(root, font=("Helvetica", 20), yscrollcommand=scrollbar.set)
output_text.pack(side=tk.LEFT, fill=tk.BOTH)

scrollbar.config(command=output_text.yview)

# Bind the "q" key to the on_closing function
root.bind("<KeyPress-q>", on_closing)

# Start the GUI
root.mainloop()

NameError: name 'tk' is not defined